# 05 Test-set audit and final refit

Workflow steps 6-8. **Run this notebook once.** It loads the test set for the first and only time, scores the single model selected in notebook 03 at the operating point fixed in notebook 04, and then refits that exact specification on all the data. If the audit fails, report the failure. Do not change the model and re-score: the test set would then become another validation set.

The test set is loaded with `load_test()`: 793 unique wines, 972 rows with their repeat measurements re-attached. This matches the grouped CV in notebook 03, where metrics were also computed over rows.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.metrics import (average_precision_score, roc_auc_score, roc_curve,
                             precision_recall_curve, confusion_matrix, accuracy_score)

from wine_utils import (load_train, load_test, load_all, transform, build_selected,
                        expected_cost, no_model_costs, FEATURES, C_FN_BASE)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

spec = json.load(open('results/selected_model.json'))
op = json.load(open('results/operating_point.json'))
assert op['model'] == spec['name']
X_train, y_train, g_train = load_train()
spec, op

## Audit pass criteria (fixed before the test set is loaded)

The audit passes only if **both** hold:
1. **Test agrees with CV:** test PR-AUC lies within the CV mean ± 2 SE from notebook 03.
2. **Beats the baselines:** test PR-AUC is above the share of good wines in the training set, test ROC-AUC is above 0.5, and test accuracy at threshold 0.5 is above the majority-class accuracy (predicting *not good* for every wine).

Operating-point results (sensitivity, specificity, expected cost and saving over the no-model policies at the threshold from notebook 04) are reported but are not pass/fail criteria.

In [ ]:
lo = spec['cv_pr_auc_mean'] - 2 * spec['cv_pr_auc_se']
hi = spec['cv_pr_auc_mean'] + 2 * spec['cv_pr_auc_se']
pr_baseline = y_train.mean()
print(f'Criterion 1: test PR-AUC in [{lo:.3f}, {hi:.3f}]')
print(f'Criterion 2: PR-AUC > {pr_baseline:.3f}, ROC-AUC > 0.5, accuracy > majority-class accuracy')

## Step 6: fit the selected specification on the full training set

In [ ]:
model = build_selected(spec).fit(X_train, y_train)
print(spec['name'], 'fitted on', len(X_train), 'rows')
model

## Step 7: score the test set, once

In [ ]:
X_test, y_test, g_test = load_test()
assert len(X_test) == 972
prob = model.predict_proba(X_test)[:, 1]

majority = int(y_train.mean() >= 0.5)
result = {
    'PR-AUC': average_precision_score(y_test, prob),
    'ROC-AUC': roc_auc_score(y_test, prob),
    'Accuracy @0.5': accuracy_score(y_test, prob >= 0.5),
    'Majority-class accuracy': accuracy_score(y_test, np.full(len(y_test), majority)),
}
passed_cv = lo <= result['PR-AUC'] <= hi
passed_base = (result['PR-AUC'] > pr_baseline and result['ROC-AUC'] > 0.5
               and result['Accuracy @0.5'] > result['Majority-class accuracy'])
print('Test rows:', len(y_test), '| unique wines:', g_test.nunique(), '| share good:', round(y_test.mean(), 3))
print('Criterion 1 (agrees with CV):', 'PASS' if passed_cv else 'FAIL')
print('Criterion 2 (beats baselines):', 'PASS' if passed_base else 'FAIL')
print('AUDIT:', 'PASSED' if passed_cv and passed_base else 'FAILED')
pd.Series(result).round(3)

### Operating point on the test set
Threshold fixed in notebook 04 (base case $c_{FN}/c_{FP} = 3$).

In [ ]:
t = op['threshold']
cm = confusion_matrix(y_test, prob >= t)
tn, fp, fn, tp = cm.ravel()
pi = y_test.mean()
nm = no_model_costs(pi, C_FN_BASE)
best_nm = min(nm, key=nm.get)
cost = expected_cost(y_test, prob, t, C_FN_BASE)

fig, ax = plt.subplots(figsize=(4, 3.5))
sns.heatmap(cm, annot=True, fmt='d', cbar=False, cmap='Blues', ax=ax,
            xticklabels=['not good', 'good'], yticklabels=['not good', 'good'])
ax.set(xlabel='Predicted', ylabel='Actual', title=f'Test set ({len(y_test)} rows), threshold {t:g}')
plt.tight_layout()
plt.show()

test_op = {
    'Sensitivity': tp / (tp + fn),
    'Specificity': tn / (tn + fp),
    'Precision': tp / (tp + fp),
    'Cost per wine': cost,
    'Cost: taste none': nm['taste none'],
    'Cost: taste all': nm['taste all'],
    'Saving vs better no-model (abs)': nm[best_nm] - cost,
    'Saving vs better no-model (%)': 100 * (1 - cost / nm[best_nm]),
}
print('Better no-model policy on the test set:', best_nm)
pd.Series(test_op).round(3)

### CV estimates against the held-out test results
CV values are the grouped 10-fold estimates on the training set (notebook 03 for PR-AUC and ROC-AUC, mean ± SE across folds; notebook 04 for the operating point, pooled out-of-fold).

In [ ]:
comparison = pd.DataFrame({
    'CV': [spec['cv_pr_auc_mean'], spec['cv_roc_auc_mean'], op['cv_sensitivity'],
           op['cv_specificity'], op['cv_precision'], op['cv_cost_per_wine'],
           op['cv_saving_abs'], op['cv_saving_pct']],
    'CV SE': [spec['cv_pr_auc_se'], spec['cv_roc_auc_se']] + [np.nan] * 6,
    'Test': [result['PR-AUC'], result['ROC-AUC'], test_op['Sensitivity'], test_op['Specificity'],
             test_op['Precision'], test_op['Cost per wine'],
             test_op['Saving vs better no-model (abs)'], test_op['Saving vs better no-model (%)']],
}, index=['PR-AUC', 'ROC-AUC', f'Sensitivity @{t:g}', f'Specificity @{t:g}', f'Precision @{t:g}',
          f'Cost per wine @{t:g}', 'Saving vs no-model (abs)', 'Saving vs no-model (%)'])
comparison['Test - CV (in SEs)'] = (comparison['Test'] - comparison['CV']) / comparison['CV SE']
comparison.round(3)

In [ ]:
fpr, tpr, _ = roc_curve(y_test, prob)
prec, rec, _ = precision_recall_curve(y_test, prob)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(fpr, tpr, lw=2, label=f"Test (AUC {result['ROC-AUC']:.3f})")
axes[0].plot([0, 1], [0, 1], color='grey', ls='--', lw=0.8, label='No skill')
axes[0].set(xlabel='False positive rate', ylabel='True positive rate', title=f'Test ROC curve, {spec["name"]}')
axes[1].plot(rec, prec, lw=2, label=f"Test (AP {result['PR-AUC']:.3f})")
axes[1].axhline(pi, color='grey', ls='--', lw=0.8, label=f'No skill ({pi:.2f})')
axes[1].set(xlabel='Recall', ylabel='Precision', title=f'Test precision-recall curve, {spec["name"]}')
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Step 8: refit the same specification on all data (4,898 rows)

Only the parameters are re-estimated. The predictors, transformations, model family and threshold stay frozen, and the performance figures to report remain the step 7 test results. Predictors are standardised over the full data, so effects are per one-SD increase.

- **Logistic:** sklearn gives no standard errors, so the same unpenalised model is refitted in `statsmodels`. Repeat measurements of a wine are not independent, so SEs are cluster-robust by `wine_id`. We report odds ratios per 1 SD.
- **LDA:** standardised discriminant coefficients (the weight of each predictor in the linear discriminant) and the standardised difference in class means.
- **Gaussian NB:** the standardised difference in class means for each predictor, which drives its per-feature likelihood ratio.

In [ ]:
df_all = load_all()
assert len(df_all) == 4898
X_all = transform(df_all[FEATURES])
y_all = (df_all['quality'] >= 7).astype(int)
g_all = df_all['wine_id']

final_model = build_selected(spec).fit(X_all, y_all)   # the deployable model
Z_all = (X_all - X_all.mean()) / X_all.std()

if spec['name'] == 'Logistic':
    final = sm.Logit(y_all, sm.add_constant(Z_all)).fit(
        disp=0, cov_type='cluster', cov_kwds={'groups': g_all})
    print(final.summary())
    ci = final.conf_int()
    effects = pd.DataFrame({
        'coef (log-odds per SD)': final.params,
        'odds ratio per SD': np.exp(final.params),
        'OR 95% CI low': np.exp(ci[0]),
        'OR 95% CI high': np.exp(ci[1]),
        'p-value': final.pvalues,
    }).drop('const').sort_values('odds ratio per SD', ascending=False)
else:
    mean_diff = Z_all[y_all == 1].mean() - Z_all[y_all == 0].mean()
    effects = pd.DataFrame({'class mean difference (good - not good, SDs)': mean_diff})
    if spec['name'] == 'LDA':
        lda = final_model.named_steps['model']
        effects['standardised discriminant coefficient'] = lda.scalings_[:, 0]
    effects = effects.sort_values(effects.columns[-1], key=np.abs, ascending=False)
effects.round(3)

## Limitations

- **Binarised outcome.** Quality is cut at 7, so the ordering within each class (a 3 vs a 6, a 7 vs a 9) is lost, and wines just either side of the cut are treated as fully different.
- **Noisy labels.** Each quality score is the median of at least three expert ratings, so some labels near the cut-off are effectively uncertain, which caps achievable accuracy.
- **One region and vintage period.** The data are Vinho Verde wines from a 2009 study (Cortez et al.). Results may not transfer to other regions, styles or later vintages.
- **Missing drivers.** There is no price, grape variety, brand or cost information, so the cost ratio $c_{FN}/c_{FP}$ is assumed, not measured (see the sensitivity analysis in notebook 04).
- **White wine only.** No conclusion is drawn about red wine.
- **CV variance.** CV estimates come from one grouped 10-fold split of 3,168 wines. The fold-to-fold SE (about 0.017 in PR-AUC) is similar in size to the gap between Logistic and LDA, so the choice between them is not clear-cut, and the test set gives one further draw of 793 wines.